# 🎯 Notebook 04 — XGBoost: Risco de Violação de OLA
## Predictfy × Locaweb — FIAP Challenge 2026

**Objetivo:** Prever a probabilidade de violação de OLA (P2 ≤4h ou P3 ≤12h) por incidente,
identificando os principais fatores de risco e gerando scores por produto e grupo de atendimento.

**Input:** `data/processed/incidents_features.parquet` (25.588 incidentes, 24 features)
**Output:** `outputs/risco_ola.json`

**Desafio:** Desbalanceamento severo — 248 violações (0.97%) vs 25.340 não-violações (99.03%)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, precision_recall_curve,
                              average_precision_score, f1_score)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import json, os
from datetime import date

print("✅ Imports ok")


In [ ]:
# ── Carregar dados ────────────────────────────────────────────────────────────
df = pd.read_parquet('../data/processed/incidents_features.parquet')
print(f"Shape: {df.shape}")
print(f"\nDistribuição do target:")
print(df['target_ola'].value_counts())
print(f"\nTaxa de violação: {df['target_ola'].mean()*100:.2f}%")


In [ ]:
# ── Features e target ─────────────────────────────────────────────────────────
FEATURES = [
    'hora', 'dia_semana', 'mes', 'trimestre',
    'is_horario_comercial', 'is_fim_de_semana', 'is_segunda_terca',
    'periodo_dia', 'prioridade_bin',
    'lag_1d', 'lag_7d', 'rolling_7d', 'rolling_30d',
    'lag_1d_p2', 'lag_1d_p3',
    'Produto_enc', 'Categoria_enc', 'Grupo designado_enc',
    'Produto_freq', 'Grupo designado_freq',
]

X = df[FEATURES]
y = df['target_ola']

# Split temporal — não pode ser aleatório em série temporal
# Usar os primeiros 80% para treino, últimos 20% para teste
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Treino: {X_train.shape} | Violações: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Teste:  {X_test.shape}  | Violações: {y_test.sum()} ({y_test.mean()*100:.2f}%)")


In [ ]:
# ── Modelo base sem balanceamento ─────────────────────────────────────────────
model_base = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=y_train.value_counts()[0] / y_train.value_counts()[1],
    eval_metric='aucpr', random_state=42, n_jobs=-1
)
model_base.fit(X_train, y_train,
               eval_set=[(X_test, y_test)],
               verbose=False)

y_pred_proba = model_base.predict_proba(X_test)[:, 1]
auc_roc  = roc_auc_score(y_test, y_pred_proba)
auc_pr   = average_precision_score(y_test, y_pred_proba)
print(f"ROC-AUC: {auc_roc:.4f}")
print(f"PR-AUC:  {auc_pr:.4f}")


In [ ]:
# ── Threshold tuning — maximizar F1 ──────────────────────────────────────────
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"Melhor threshold: {best_threshold:.3f}")
print(f"F1 máximo: {f1_scores[best_idx]:.4f}")
print(f"Precision: {precision[best_idx]:.4f}")
print(f"Recall:    {recall[best_idx]:.4f}")

y_pred_tuned = (y_pred_proba >= best_threshold).astype(int)
print("\nClassification Report (threshold otimizado):")
print(classification_report(y_test, y_pred_tuned, target_names=['Não Viola', 'Viola']))


In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
importances = pd.DataFrame({
    'feature': FEATURES,
    'importance': model_base.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 features mais importantes:")
print(importances.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances.head(12)['feature'][::-1],
        importances.head(12)['importance'][::-1], color='#E8002D')
ax.set_xlabel('Importance')
ax.set_title('XGBoost — Feature Importance (Top 12)')
plt.tight_layout()
plt.savefig('../outputs/xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico salvo")


In [ ]:
# ── Scores de risco por produto e grupo ───────────────────────────────────────
# Calcular probabilidade média de violação por produto e grupo no dataset completo
df['prob_violacao'] = model_base.predict_proba(X)[:, 1]

# Carregar mapeamento reverso (enc → nome original)
raw = pd.read_excel('../data/raw/LW-DATASET.xlsx')
kpi = raw[raw['Entrou para KPI?']=='SIM'].copy()

# Mapear enc de volta para nome
prod_map  = dict(zip(df['Produto_enc'],   kpi['Produto'].values[:len(df)]))
grupo_map = dict(zip(df['Grupo designado_enc'], kpi['Grupo designado'].values[:len(df)]))

# Score por produto
score_prod = df.groupby('Produto_enc').agg(
    total=('target_ola','count'),
    violacoes=('target_ola','sum'),
    prob_media=('prob_violacao','mean')
).reset_index()
score_prod['taxaViolacao'] = (score_prod['violacoes'] / score_prod['total'] * 100).round(2)
score_prod['probViolacao'] = (score_prod['prob_media'] * 100).round(1)
score_prod = score_prod.sort_values('probViolacao', ascending=False)

print("Top 5 produtos por risco:")
print(score_prod.head(5)[['Produto_enc','total','violacoes','taxaViolacao','probViolacao']].to_string(index=False))


In [ ]:
# ── Exportar risco_ola.json ───────────────────────────────────────────────────
# Score por grupo
score_grupo = df.groupby('Grupo designado_enc').agg(
    total=('target_ola','count'),
    violacoes=('target_ola','sum'),
    prob_media=('prob_violacao','mean')
).reset_index()
score_grupo['taxaViolacao'] = (score_grupo['violacoes'] / score_grupo['total'] * 100).round(2)
score_grupo['probViolacao'] = (score_grupo['prob_media'] * 100).round(1)
score_grupo = score_grupo.sort_values('probViolacao', ascending=False)

output = {
    "modelo": "xgboost_ola_risk",
    "gerado_em": date.today().strftime('%Y-%m-%d'),
    "metricas": {
        "roc_auc": round(auc_roc, 4),
        "pr_auc":  round(auc_pr, 4),
        "threshold_otimizado": round(best_threshold, 3),
        "f1_violacao": round(f1_scores[best_idx], 4),
    },
    "produtos": score_prod.head(10).rename(columns={'Produto_enc':'id'})[
        ['id','total','violacoes','taxaViolacao','probViolacao']
    ].to_dict('records'),
    "grupos": score_grupo.head(10).rename(columns={'Grupo designado_enc':'id'})[
        ['id','total','violacoes','taxaViolacao','probViolacao']
    ].to_dict('records'),
    "top_features": importances.head(10)[['feature','importance']].assign(
        importance=lambda d: d['importance'].round(4)
    ).to_dict('records'),
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/risco_ola.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ risco_ola.json exportado")
print(json.dumps(output, ensure_ascii=False, indent=2)[:800])
